# Foundations of Big Data Analytics with Python (FBDA)
## Dynamic Analytical Dashboard — ShopEase Order Data

**Project Title:** Dynamic Analytical Dashboard with Python — ShopEase Order Data
**Student Names:** Shibalik Dey, Vaibhav Singh, Ayush Raj
**Enrollment Numbers:** ...075041, ...075049, ...075018
**Group ID:** 018_041_049 &nbsp;|&nbsp; **Fixed Randomization Seed:** 18041049
**Live Data Link:** https://www.kaggle.com/datasets/mernahabib/shopease-raw-orders-dataset

This notebook walks through the full analysis behind our Streamlit dashboard and project
report: loading the raw ShopEase orders data, drawing our fixed-seed sample, cleaning it,
and then running the descriptive, inferential, and causal (regression) analysis required
by the project brief.


## 0. Setup

In [ ]:
# Run this once per Colab session
!pip install -q pandas numpy matplotlib seaborn scipy statsmodels


In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

GROUP_ID = "018_041_049"
SEED = 18041049          # Group ID without underscores, per the project brief
SAMPLE_SIZE = 10001

random.seed(SEED)
np.random.seed(SEED)
print(f"Group ID: {GROUP_ID} | Seed: {SEED}")


## 1. Load the Raw Data

**Data source (live link):** https://www.kaggle.com/datasets/mernahabib/shopease-raw-orders-dataset

Kaggle requires authentication to download programmatically, so the simplest path in
Colab is to upload the CSV directly (Kaggle → Download → upload the extracted file here).
If you'd rather pull it via the Kaggle API, use the commented block below instead.

In [ ]:
from google.colab import files

# Option A — manual upload (simplest): run this cell and choose the CSV
uploaded = files.upload()  # select shopease_raw_orders.csv
DATA_PATH = list(uploaded.keys())[0]

# Option B — Kaggle API (uncomment if you have a kaggle.json API token):
# from google.colab import files
# files.upload()  # upload your kaggle.json here
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d mernahabib/shopease-raw-orders-dataset -p . --unzip
# DATA_PATH = "shopease_raw_orders.csv"

raw_df = pd.read_csv(DATA_PATH)
print("Raw shape:", raw_df.shape)
raw_df.head()


In [ ]:
raw_df.info()


## 2. Fixed-Randomization Sampling

Per the project brief, we draw a fixed sample of **10,001 records** using
`random_state = 18041049` (our Group ID `018_041_049` without underscores).

In [ ]:
replace = len(raw_df) < SAMPLE_SIZE
sampled_df = raw_df.sample(n=SAMPLE_SIZE, random_state=SEED, replace=replace).reset_index(drop=True)
print("Sampled shape:", sampled_df.shape)
sampled_df.head()


## 3. Data Cleaning

The raw export is genuinely messy: inconsistent casing/whitespace in text fields, discount
values mixed between decimal fractions and percentage strings, implausible ages/quantities/
prices/ratings, mixed date formats, and duplicate rows/OrderIDs. We fix all of this before
trusting any downstream number.

In [ ]:
def strip_col(series):
    return series.astype(str).str.strip()

def parse_discount(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s.endswith("%"):
        try:
            return float(s.replace("%", "")) / 100
        except ValueError:
            return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

df = sampled_df.copy()

# --- standardize text fields ---
for col in ["Gender", "City", "Category", "Product", "PaymentMethod", "OrderStatus"]:
    df[col] = strip_col(df[col])
    df[col] = df[col].replace({"nan": np.nan})

gender_map = {"f": "Female", "female": "Female", "m": "Male", "male": "Male"}
df["Gender"] = df["Gender"].str.lower().map(gender_map).fillna(df["Gender"])

df["City"] = df["City"].str.title()
df["Category"] = df["Category"].str.title()
df["Product"] = df["Product"].str.title()

payment_map = {
    "bank transfer": "Bank Transfer", "wallet": "Wallet", "cash": "Cash",
    "visa": "Visa", "mastercard": "Mastercard",
}
df["PaymentMethod"] = df["PaymentMethod"].str.lower().map(payment_map).fillna(df["PaymentMethod"])

status_map = {
    "delivered": "Delivered", "pending": "Pending",
    "cancelled": "Cancelled", "canceled": "Cancelled",
}
df["OrderStatus"] = df["OrderStatus"].str.lower().map(status_map).fillna(df["OrderStatus"])

# --- discount: unify mixed formats ---
df["Discount"] = df["Discount"].apply(parse_discount)

# --- dates ---
df["OrderDate"] = pd.to_datetime(df["OrderDate"], errors="coerce")
df["DeliveryDate"] = pd.to_datetime(df["DeliveryDate"], errors="coerce")

# --- implausible values -> missing ---
df.loc[(df["CustomerAge"] < 10) | (df["CustomerAge"] > 100), "CustomerAge"] = np.nan
df.loc[df["Quantity"] <= 0, "Quantity"] = np.nan
df.loc[df["UnitPrice"] <= 0, "UnitPrice"] = np.nan
df.loc[(df["Rating"] < 1) | (df["Rating"] > 5), "Rating"] = np.nan

# --- recompute TotalAmount consistently ---
df["TotalAmount"] = (df["UnitPrice"] * df["Quantity"] * (1 - df["Discount"].fillna(0))).round(2)

# --- drop duplicates ---
before = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset="OrderID", keep="first").reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicate rows -> final n = {len(df)}")

df.head()


In [ ]:
NUMERIC_COLS = ["CustomerAge", "Quantity", "UnitPrice", "Discount", "Rating", "TotalAmount"]
CATEGORICAL_COLS = ["Gender", "City", "Category", "Product", "PaymentMethod", "OrderStatus"]

print("Missing values after cleaning:")
df[NUMERIC_COLS].isna().sum()


## 4. Descriptive Statistics

### 4.1 Non-Categorical (Numeric) Variables

In [ ]:
desc = df[NUMERIC_COLS].describe(percentiles=[.25, .5, .75]).T
desc["skewness"] = df[NUMERIC_COLS].skew()
desc["kurtosis"] = df[NUMERIC_COLS].kurt()
desc["mode"] = df[NUMERIC_COLS].mode().iloc[0]
desc.round(2)


### 4.2 Categorical Variables

In [ ]:
for col in CATEGORICAL_COLS:
    vc = df[col].value_counts(dropna=True)
    rel = df[col].value_counts(normalize=True, dropna=True).round(4)
    table = pd.DataFrame({"Frequency": vc, "Relative Frequency": rel})
    print(f"--- {col} ---")
    print(table)
    print(f"Highest: {vc.idxmax()} ({vc.max()}) | Lowest: {vc.idxmin()} ({vc.min()})\n")


## 4.3 Visualizations — Non-Categorical Data

In [ ]:
# Scatter plot
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df, x="UnitPrice", y="TotalAmount", hue="Category", alpha=0.6, ax=ax)
ax.set_title("UnitPrice vs TotalAmount by Category")
plt.tight_layout()
plt.show()


In [ ]:
# Line plot — weekly revenue trend
trend = df.dropna(subset=["OrderDate"]).set_index("OrderDate").resample("W")["TotalAmount"].sum()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(trend.index, trend.values, marker="o", color="darkorange")
ax.set_title("Weekly Revenue Trend")
ax.set_ylabel("Revenue (EGP)")
plt.tight_layout()
plt.show()


In [ ]:
# Box-Whisker plot
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x="Category", y="TotalAmount", showfliers=False, ax=ax)
ax.set_title("Order Value by Category (outliers hidden)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# Violin plot
fig, ax = plt.subplots(figsize=(9, 5))
sns.violinplot(data=df, x="Category", y="Rating", ax=ax)
ax.set_title("Rating Distribution by Category")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# Heat map — correlation
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(df[NUMERIC_COLS].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Pearson Correlation Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# Pair plot
sns.pairplot(df[["UnitPrice", "Quantity", "TotalAmount"]].dropna())
plt.show()


## 4.4 Visualizations — Categorical Data

In [ ]:
# Bar plot
vc = df["Category"].value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=vc.values, y=vc.index, hue=vc.index, palette="crest", legend=False, ax=ax)
ax.set_title("Order Count by Category")
plt.tight_layout()
plt.show()


In [ ]:
# Histogram
trimmed = df["TotalAmount"][df["TotalAmount"] < df["TotalAmount"].quantile(0.99)]
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(trimmed.dropna(), bins=40, kde=True, color="teal", ax=ax)
ax.set_title("Distribution of Order Value (below 99th percentile)")
plt.tight_layout()
plt.show()


In [ ]:
# Pie plot
status_counts = df["OrderStatus"].value_counts()
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(status_counts.values, labels=status_counts.index, autopct="%1.1f%%", colors=sns.color_palette("pastel"))
ax.set_title("Order Status Distribution")
plt.tight_layout()
plt.show()


## 5. Inferential Statistics

### 5.1 Confidence Interval for the Mean

In [ ]:
series = df["TotalAmount"].dropna()
mean, sem = series.mean(), stats.sem(series)
ci = stats.t.interval(0.95, len(series) - 1, loc=mean, scale=sem)
print(f"Mean TotalAmount = {mean:.2f}")
print(f"95% CI = ({ci[0]:.2f}, {ci[1]:.2f})")


### 5.2 Test of Mean — Independent t-test (Gender) & One-Way ANOVA (Category)

In [ ]:
male = df.loc[df.Gender == "Male", "TotalAmount"].dropna()
female = df.loc[df.Gender == "Female", "TotalAmount"].dropna()
t_stat, p_val = stats.ttest_ind(male, female, equal_var=False)
print(f"t-test (Male vs Female TotalAmount): t = {t_stat:.3f}, p = {p_val:.4f}")

groups = [g.dropna().values for _, g in df.groupby("Category")["TotalAmount"]]
f_stat, p_anova = stats.f_oneway(*groups)
print(f"ANOVA (TotalAmount across Category): F = {f_stat:.2f}, p = {p_anova:.6f}")


### 5.3 Test of Variance — Levene's Test

In [ ]:
lev_stat, lev_p = stats.levene(*groups)
print(f"Levene's test: W = {lev_stat:.2f}, p = {lev_p:.6f}")


### 5.4 Test of Normality — Shapiro-Wilk & Kolmogorov-Smirnov

In [ ]:
sample_for_test = df["TotalAmount"].dropna().sample(min(5000, df["TotalAmount"].notna().sum()), random_state=SEED)
sh_stat, sh_p = stats.shapiro(sample_for_test)
print(f"Shapiro-Wilk: W = {sh_stat:.4f}, p = {sh_p:.6f}")

ks_stat, ks_p = stats.kstest(stats.zscore(sample_for_test), "norm")
print(f"Kolmogorov-Smirnov: D = {ks_stat:.4f}, p = {ks_p:.6f}")


### 5.5 Chi-Square Test of Independence

In [ ]:
for c1, c2 in [("Category", "OrderStatus"), ("Gender", "OrderStatus")]:
    contingency = pd.crosstab(df[c1], df[c2])
    chi2, p, dof, _ = stats.chi2_contingency(contingency)
    print(f"{c1} vs {c2}: chi2 = {chi2:.2f}, p = {p:.4f}, dof = {dof}")


### 5.6 Non-Parametric Test — Mann-Whitney U

In [ ]:
m_rating = df.loc[df.Gender == "Male", "Rating"].dropna()
f_rating = df.loc[df.Gender == "Female", "Rating"].dropna()
u_stat, u_p = stats.mannwhitneyu(m_rating, f_rating)
print(f"Mann-Whitney U (Rating, Male vs Female): U = {u_stat:.1f}, p = {u_p:.4f}")


## 6. Causal Analysis — Regression

### 6.1 Multiple Linear Regression (with a categorical predictor)

In [ ]:
reg_df = df[["TotalAmount", "UnitPrice", "Quantity", "Discount", "Category"]].dropna()
model = smf.ols("TotalAmount ~ UnitPrice + Quantity + Discount + C(Category)", data=reg_df).fit()
print(model.summary())


### 6.2 Polynomial Regression

In [ ]:
poly_df = df[["UnitPrice", "TotalAmount"]].dropna()
coeffs = np.polyfit(poly_df["UnitPrice"], poly_df["TotalAmount"], deg=2)
poly_fn = np.poly1d(coeffs)

x_sorted = np.linspace(poly_df["UnitPrice"].min(), poly_df["UnitPrice"].max(), 200)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(poly_df["UnitPrice"], poly_df["TotalAmount"], alpha=0.3, label="Data")
ax.plot(x_sorted, poly_fn(x_sorted), color="red", label="Degree 2 fit")
ax.set_xlabel("UnitPrice")
ax.set_ylabel("TotalAmount")
ax.legend()
plt.tight_layout()
plt.show()


### 6.3 Logistic Regression — Predicting Order Cancellation

In [ ]:
logit_df = df[["UnitPrice", "Quantity", "Discount", "OrderStatus"]].dropna()
logit_df["Cancelled"] = (logit_df["OrderStatus"] == "Cancelled").astype(int)
X = sm.add_constant(logit_df[["UnitPrice", "Quantity", "Discount"]])
y = logit_df["Cancelled"]
logit_model = sm.Logit(y, X).fit()
print(logit_model.summary())


## 7. Key Findings (Summary)

- **Electronics drives revenue disproportionately**: ~19.8% of orders but ~49.8% of total revenue.
- **Order value is heavily right-skewed** — the median (~1,600 EGP) is a more representative
  "typical order" than the mean (~3,559 EGP).
- **Gender doesn't meaningfully affect order value, rating, or cancellation** (t-test, chi-square,
  Mann-Whitney all non-significant), but **Category does** (ANOVA is highly significant).
- **Price and quantity explain ~76% of order-value variation** (R² = 0.756); once they're
  accounted for, Category itself adds no further explanatory power.
- **Cancellation isn't predictable from price, quantity, or discount** (logistic regression
  Pseudo R² ≈ 0.0003) — the driver is likely operational (delivery, stock, payment issues),
  not pricing.
- **Cairo is the dominant market** (37.6% of orders); Aswan is the smallest (5.8%).

See the full write-up, managerial recommendations, and individual contributions in
`FBDA_Analytical_Report.docx`.


## 8. (Optional) Save the Cleaned Data

In [ ]:
df.to_csv("shopease_cleaned.csv", index=False)
print("Saved shopease_cleaned.csv:", df.shape)


## 9. (Optional) Launch the Streamlit Dashboard from Colab

Streamlit doesn't run natively inside a Colab cell, so we expose it with a tunnel.

**Why the old version showed `502 Bad Gateway`:** the tunnel was opened after a fixed
`sleep(5)` even if Streamlit had crashed or hadn't finished starting (missing `app.py` /
CSV, missing package, slow start), and `localtunnel` can also try `localhost` over IPv6 while
Streamlit only listens on IPv4. The cells below fix this by checking the files, starting
Streamlit on `127.0.0.1`, **waiting until it is actually healthy**, printing its log if it
fails, and only then opening the tunnel.

Run the cells **in order**. Upload `app.py` and your CSV to the Colab working directory first
(Files pane). If `app.py` is missing, cell 9.2 creates a basic working dashboard for you.

In [ ]:
# 9.1 Install (run once per session)
!pip install -q streamlit
!npm install -g localtunnel -q

In [ ]:
# 9.2 Make sure the files Streamlit needs exist
import os

print("Working dir:", os.getcwd())
print("Files:", [f for f in os.listdir(".") if not f.startswith(".")])

# Save the cleaned data so the app always has a CSV to read
if "df" in globals():
    df.to_csv("shopease_cleaned.csv", index=False)
    print("Saved shopease_cleaned.csv", df.shape)

# Fallback dashboard - only written if you did NOT upload your own app.py
if not os.path.exists("app.py"):
    FALLBACK_APP = '''
import os
import pandas as pd
import streamlit as st
import plotly.express as px

st.set_page_config(page_title="ShopEase Dashboard", layout="wide")
st.title("ShopEase Order Analytics Dashboard")

@st.cache_data
def load():
    path = "shopease_cleaned.csv" if os.path.exists("shopease_cleaned.csv") else "shopease_raw_orders.csv"
    d = pd.read_csv(path)
    d["OrderDate"] = pd.to_datetime(d["OrderDate"], errors="coerce")
    return d

df = load()

st.sidebar.header("Filters")
cats = st.sidebar.multiselect("Category", sorted(df["Category"].dropna().unique()),
                              default=sorted(df["Category"].dropna().unique()))
cities = st.sidebar.multiselect("City", sorted(df["City"].dropna().unique()),
                                default=sorted(df["City"].dropna().unique()))
f = df[df["Category"].isin(cats) & df["City"].isin(cities)]

c1, c2, c3 = st.columns(3)
c1.metric("Orders", f"{len(f):,}")
c2.metric("Revenue (EGP)", f"{f['TotalAmount'].sum():,.0f}")
c3.metric("Avg Rating", f"{f['Rating'].mean():.2f}")

a, b = st.columns(2)
a.plotly_chart(px.bar(f.groupby("Category")["TotalAmount"].sum().reset_index(),
                      x="Category", y="TotalAmount", title="Revenue by Category"),
               use_container_width=True)
b.plotly_chart(px.pie(f, names="OrderStatus", title="Order Status"),
               use_container_width=True)
weekly = f.dropna(subset=["OrderDate"]).set_index("OrderDate").resample("W")["TotalAmount"].sum().reset_index()
st.plotly_chart(px.line(weekly, x="OrderDate", y="TotalAmount", title="Weekly Revenue"),
                use_container_width=True)
st.dataframe(f.head(200))
'''
    open("app.py", "w").write(FALLBACK_APP)
    print("app.py was missing -> created a basic fallback app.py")
else:
    print("Using your existing app.py")

# The app needs a data file. Warn early instead of failing later with a 502.
if not (os.path.exists("shopease_cleaned.csv") or os.path.exists("shopease_raw_orders.csv")):
    print("WARNING: no CSV found - upload shopease_raw_orders.csv or run the cleaning cells first.")

In [ ]:
# 9.3 Start Streamlit and WAIT until it is healthy (this is what prevents the 502)
import subprocess, time, urllib.request, os, signal

PORT = 8501

# Stop any old Streamlit process left over from a previous run
!pkill -f "streamlit run" 2>/dev/null
time.sleep(2)

# If your app.py imports extra packages (plotly, etc.) make sure they are installed
!pip install -q plotly

log = open("streamlit.log", "w")
proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.headless", "true",
     "--server.port", str(PORT),
     "--server.address", "127.0.0.1",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    stdout=log, stderr=subprocess.STDOUT,
)

ready = False
for i in range(60):                       # wait up to ~60 seconds
    if proc.poll() is not None:           # process died -> stop waiting
        break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/_stcore/health", timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(1)

if ready:
    print(f"Streamlit is UP on port {PORT}. Now run the next cell to open the tunnel.")
else:
    print("Streamlit did NOT start. This is the real cause of the 502. Log:\n")
    print(open("streamlit.log").read())

In [ ]:
# 9.4 Open the tunnel (only run this after 9.3 says "Streamlit is UP")
import urllib.request

# Tunnel password = this machine's public IP
ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip()
print("Tunnel password (paste it on the localtunnel page):", ip)

# --local-host 127.0.0.1 avoids the IPv6 'localhost' problem that also causes 502
!npx localtunnel --port 8501 --local-host 127.0.0.1

### If localtunnel still gives 502 or won't load

`localtunnel` is often flaky. Cloudflare's quick tunnel is more reliable and needs no password.
Run **9.3 first**, then this cell instead of 9.4:

In [ ]:
# Alternative to 9.4: Cloudflare quick tunnel (more reliable, no password page)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import subprocess, re, time
tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8501"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 30:
    line = tun.stdout.readline()
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0); break
print("Open your dashboard here:", url or "no URL yet - re-run this cell")